#  Step 1: Install Dependencies
We will install the `ultralytics` package to get access to YOLO11, and `roboflow` to download your custom images later.

In [ ]:
!pip install -q ultralytics roboflow
!pip install -q fiftyone

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 136.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:0

Now, let's import the necessary modules and verify that we have access to a GPU runtime, which will speed up our training significantly.

In [ ]:
import torch
import ultralytics
#from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Ultralytics version: 8.4.183
CUDA available: True
Active GPU: Tesla T4


# Step 2: Download Your Custom Roboflow Dataset
Use the snippet below to load your dataset.
Make sure to replace `YOUR_API_KEY`, `workspace-name`, `project-name`, and `version-number` with your actual Roboflow details.

In [ ]:
from roboflow import Roboflow

# Initialize Roboflow client
rf = Roboflow(api_key="GOXcbY3plelrniCVW3q4")

# Leaving workspace() empty automatically selects your default workspace
project = rf.workspace().project("abdulghani")
version = project.version(4) # Change version number if needed

# Download dataset in YOLO format
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to abdulghani-4 in yolov11:: 100%|██████████| 20682/20682 [00:04<00:00, 4599.06it/s] 


# Step 3: Configure Dataset and Merge Classes with COCO
Here we will define our target classes, and downlowd the classes from coco (COCO classes + your custom classes) and write a custom YAML configuration file for YOLO11 training.

In [ ]:
%pip uninstall -y fiftyone

Found existing installation: fiftyone 1.22.1
Uninstalling fiftyone-1.22.1:
  Successfully uninstalled fiftyone-1.22.1


In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in ["fiftyone", "Pillow", "ultralytics", "numpy"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: غير مثبت")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
fiftyone: غير مثبت
Pillow: 13.0.0
ultralytics: 8.4.183
numpy: 2.1.3


In [ ]:
%pip install fiftyone==1.22.1

  Using cached fiftyone-1.22.1-py3-none-any.whl.metadata (23 kB)
Using cached fiftyone-1.22.1-py3-none-any.whl (24.0 MB)


In [ ]:
import os
import glob
import shutil
import yaml
from collections import Counter

# ==================================================
# 1. المسارات
# ==================================================
source_dir = "/content/abdulghani-4"
coco_dir = "/content/coco_subset"
output_dir = "/content/traffic_merged"

# ترتيب الفئات الأصلي في Roboflow
with open(os.path.join(source_dir, "data.yaml"), "r") as f:
    rf_config = yaml.safe_load(f)

rf_names = rf_config["names"]

if isinstance(rf_names, list):
    rf_names = dict(enumerate(rf_names))
else:
    rf_names = {int(k): v for k, v in rf_names.items()}

# استخدام أسماء الفئات الأصلية وترتيبها
class_names = {
    i: rf_names[i]
    for i in sorted(rf_names)
}

name_to_id = {
    name.lower(): class_id
    for class_id, name in class_names.items()
}

print("Final classes:", class_names)

# ==================================================
# 2. التحقق من الفئات المطلوبة
# ==================================================
required_classes = {
    "car", "motorcycle", "truck", "bus",
    "van", "policemotorsycle"
}

missing = required_classes - set(name_to_id)

if missing:
    raise ValueError(
        f"Required classes are missing from Roboflow YAML: {missing}"
    )

# ==================================================
# 3. إنشاء مجلد جديد
# ==================================================
if os.path.exists(output_dir):
    raise FileExistsError(
        f"{output_dir} already exists. "
        "Rename or remove it only after checking its contents."
    )

for split in ["train", "valid", "test"]:
    os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)

# ==================================================
# 4. نسخ بيانات Roboflow الأصلية
# ==================================================
image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]

def list_images(directory):
    results = []
    for ext in image_extensions:
        results.extend(
            glob.glob(os.path.join(directory, ext))
        )
    return sorted(results)

def copy_roboflow_split(split):
    src_images = os.path.join(source_dir, split, "images")
    src_labels = os.path.join(source_dir, split, "labels")

    if not os.path.isdir(src_images) or not os.path.isdir(src_labels):
        print(f"Skipping missing Roboflow split: {split}")
        return

    copied = 0

    for img in list_images(src_images):
        stem = os.path.splitext(os.path.basename(img))[0]
        label_path = os.path.join(src_labels, stem + ".txt")

        # نتأكد من وجود ملف التسمية المقابل
        if not os.path.isfile(label_path):
            print("Warning: missing label:", img)
            continue

        shutil.copy2(
            img,
            os.path.join(output_dir, split, "images", os.path.basename(img))
        )

        shutil.copy2(
            label_path,
            os.path.join(output_dir, split, "labels", stem + ".txt")
        )

        copied += 1

    print(f"Roboflow {split}: copied {copied} images")

copy_roboflow_split("train")
copy_roboflow_split("valid")
copy_roboflow_split("test")

# ==================================================
# 5. إضافة بيانات COCO مع إعادة ترقيم الفئات
# ==================================================
# ترتيب classes في عملية تصدير COCO السابقة:
coco_classes = ["motorcycle", "car", "truck", "bus"]

# COCO temporary ID -> final Roboflow ID
remap = {
    temporary_id: name_to_id[class_name]
    for temporary_id, class_name in enumerate(coco_classes)
}

print("COCO class remapping:")
for temporary_id, final_id in remap.items():
    print(
        f"{temporary_id}: {coco_classes[temporary_id]}"
        f" -> {final_id}: {class_names[final_id]}"
    )

def merge_coco_split(coco_split, target_split):
    src_images = f"{coco_dir}/images/{coco_split}"
    src_labels = f"{coco_dir}/labels/{coco_split}"

    if not os.path.isdir(src_images) or not os.path.isdir(src_labels):
        raise FileNotFoundError(
            f"Missing COCO image/label directory for {coco_split}"
        )

    dst_images = f"{output_dir}/{target_split}/images"
    dst_labels = f"{output_dir}/{target_split}/labels"

    added = 0

    for img in list_images(src_images):
        stem = os.path.splitext(os.path.basename(img))[0]
        label_path = os.path.join(src_labels, stem + ".txt")

        if not os.path.isfile(label_path):
            continue

        new_lines = []

        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if not parts:
                    continue

                temporary_id = int(parts[0])

                if temporary_id not in remap:
                    raise ValueError(
                        f"Unexpected COCO class ID {temporary_id} "
                        f"in {label_path}"
                    )

                parts[0] = str(remap[temporary_id])
                new_lines.append(" ".join(parts))

        # لا نضيف ملفًا لا يحتوي على أي كائن
        if not new_lines:
            continue

        # Prefix يمنع تصادم أسماء الملفات مع Roboflow
        new_image_name = "coco_" + os.path.basename(img)
        new_label_name = "coco_" + stem + ".txt"

        shutil.copy2(
            img,
            os.path.join(dst_images, new_image_name)
        )

        with open(
            os.path.join(dst_labels, new_label_name), "w"
        ) as f:
            f.write("\n".join(new_lines) + "\n")

        added += 1

    print(f"COCO {coco_split} -> {target_split}: added {added} images")

merge_coco_split("train", "train")
merge_coco_split("val", "valid")

# ==================================================
# 6. إنشاء ملف YAML الجديد
# ==================================================
yaml_data = {
    "path": output_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": class_names
}

yaml_path = "/content/traffic_merged.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print("\nNew dataset YAML:", yaml_path)

Final classes: {0: 'bus', 1: 'car', 2: 'motorcycle', 3: 'policemotorsycle', 4: 'truck', 5: 'van'}
Roboflow train: copied 9627 images
Roboflow valid: copied 455 images
Roboflow test: copied 257 images
COCO class remapping:
0: motorcycle -> 2: motorcycle
1: car -> 1: car
2: truck -> 4: truck
3: bus -> 0: bus
COCO train -> train: added 3000 images
COCO val -> valid: added 500 images

New dataset YAML: /content/traffic_merged.yaml


# Step 4: Initialize and Train YOLO11 Model
Now, we initialize a pre-trained YOLO11 model and start training using our custom YAML file configuration.

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Ultralytics version: 8.4.183
PyTorch version: 2.11.0+cu130
CUDA available: True
Active GPU: Tesla T4


In [ ]:
# Load a pre-trained YOLO11 nano detection model
model = YOLO('yolo11n.pt')

# Start training
results = model.train(
    data="/content/traffic_merged.yaml",  # Path to our custom dataset config
    epochs=25,              # Number of epochs (adjust based on needs)
    imgsz=640,              # Image size
    batch=16,               # Batch size
    device=0 if torch.cuda.is_available() else 'cpu', # Train on GPU if available
    project='yolo11_custom_coco_training',
    name='run_1',

    #freeze=10,                # تجميد أول 10 طبقات (الـ backbone)
    optimizer='AdamW',        # تحديد الـ optimizer صراحة، وإلا يُتجاهل lr0
    lr0=0.001,                # معدل تعلم أقل للحفاظ على ما تعلمه النموذج
    patience=15,              # إيقاف مبكر إذا لم تتحسن النتائج
)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/traffic_merged.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_1-3, nbs=64, nms=None, opset=None, optimize=False, opti